[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/connections/connections.ipynb)

# Connections on Curved Surfaces

This example computes parallel transport, trivial connections, and globally smooth tangent vector and stripe fields on curved surfaces. Carrying a tangent direction along a curved surface while keeping it tangent and turning it only as the surface bends defines parallel transport, whose closed-loop rotation measures intrinsic curvature and whose diffusion and low-energy eigenmodes propagate and smooth direction fields globally. Adding an extra tangent-plane turn on each edge of a triangulated surface cancels triangle and handle holonomies so that parallel fields acquire prescribed singularities consistent with the surface's Euler characteristic, and advancing a directional phase across those edges produces surface-filling stripe and cross-stripe patterns that fork automatically where uniform spacing cannot continue.

Joining two boundary edges of each triangle `edges[0] & edges[1]` represents surface normals as intrinsic tangent-plane patches, so the shortest dihedral rotor carrying `tail` onto `head` is `(head * tail).square_root()`, which rotates in the plane of the two normals while fixing the intersection line of their tangent planes. Around any closed loop of patches, squaring the cumulative rotor reads the signed holonomy angle `(loop_rotor * loop_rotor).log() | patch.dual()` directly in the tangent-plane bivector `patch.dual()`. Applying endpoint rotors across edge boundaries `(spdiag(transports >> Point) * topology.boundary[1, 0]).sum(axis=0)` differentiates three-dimensional tangent directions directly in ambient space, and pulling cotangent-weighted dual pairings back via `covariant.adjugate()((weights * Point.dual())(covariant))` forms the connection Laplacian while keeping tangent and normal components decoupled.

In the local polar-frame notation of discrete differential geometry, as laid out by [Crane, Desbrun, and Schröder (2010)](https://geometry.caltech.edu/pubs/CDS10.pdf) and [Knöppel, Crane, Pinkall, and Schröder (2015)](https://www.cs.cmu.edu/~kmcrane/Projects/StripePatterns/), each vertex $i$ picks an arbitrary unit reference tangent vector $\mathbf e_i$ to encode tangent directions as complex scalars $u_i \in \mathbb C$, rescales corner angles so polar coordinates $\varphi_{ij} = \operatorname{atan2}((\mathbf n_i \times \mathbf e_i)\cdot \tilde{\mathbf e}_{ij}, \mathbf e_i \cdot \tilde{\mathbf e}_{ij})$ sum to $2\pi$ around the vertex defect, unfolds edge-sharing triangles into a common plane to measure frame transition angles $\rho_{ij}$, and assembles the connection Laplacian as a Hermitian matrix $(L u)_i = \sum_{j \sim i} w_{ij}\bigl(u_i - e^{\mathrm i(\rho_{ij} + \theta_{ij})} u_j\bigr)$.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology
from examples.surfaces.connections import render
from examples.surfaces.connections.scenarios import face_indices, nearest_faces
from examples.util import shapes
from examples.util.animation import save_figure, show_states

context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()                                  # Scalar+
# Surface positions and tangent directions are intrinsic antivectors.
Point = ga.gatype.antivector()                               # Antivector-
Plane = ga.gatype.vector()                                   # Vector+
# Unit surface normals are intrinsic tangent-plane patches formed by joining two edge directions.
Patch = Plane.derive.flipped_orientation                     # Vector-
Rotor = ga.gatype.rotor()                                    # Rotor+

frame_duration_ms = 40

## Holonomy around latitude circles

On the unit sphere, parallel transport along a curve turns a tangent direction at each step by the shortest rotor `(head * tail).square_root()` between consecutive unit surface patches. Because each step turns only in the plane spanned by the two patches, the carried vector remains tangent throughout and returns around a closed latitude circle rotated in its starting tangent plane by the solid angle of the spherical cap enclosed by the loop.

In [ ]:
# Subdivision level of the sphere and colatitudes (30, 60, and 90 degrees) of its three latitude rings.
n_sphere_levels = 3
colatitudes = np.array([np.pi / 6, np.pi / 3, np.pi / 2])
n_ring_steps = 210
ring_stride = 2

In [ ]:
def latitude_rings(colatitudes: np.ndarray, n_ring_steps: int) -> tuple[Patch, Point]:
    """Rotating a north-pole patch and meridian tangent to `colatitudes` (`[n_rings]`) and sweeping
    around the vertical axis over `n_ring_steps` segments forms closed latitude loops of unit
    surface patches (`[n_ring_steps + 1, n_rings] Patch-`) and initial meridian directions
    (`[n_rings] Point-`)."""
    longitudes = np.linspace(0, 2 * np.pi, n_ring_steps + 1)
    tilt_rotors = (mv.zx * (-colatitudes / 2)).exp()                                      # [n_rings] Rotor+
    spin_rotors = (mv.xy * (-longitudes[:, None] / 2)).exp()                              # [n_ring_steps + 1, 1] Rotor+
    tilted_patches = tilt_rotors >> mv.xy.dual()                                          # [n_rings] Patch-
    patches = spin_rotors >> tilted_patches                                               # [n_ring_steps + 1, n_rings] Patch-
    starts = tilt_rotors >> mv.x.dual()                                                   # [n_rings] Point-
    return patches, starts


def parallel_transport(patches: Patch, start: Point) -> Iterator[Point]:
    """Yields `start` (`[...] Point-`) carried step by step along `patches` (`[n_steps + 1, ...] Patch-`)
    by the shortest dihedral rotor between consecutive unit surface patches."""
    yield start
    for tail, head in zip(patches[:-1], patches[1:]):
        turn = (head * tail).square_root()                                                # [...] Rotor+
        start = turn >> start                                                             # [...] Point-
        yield start


sphere_topology, sphere_coordinates = shapes.sphere(n_sphere_levels)
V_sphere, E_sphere, F_sphere = sphere_topology.n_elements
sphere_vertices = mv.vector(sphere_coordinates).dual().field()                            # Point-[V_sphere]
rings, starts = latitude_rings(colatitudes, n_ring_steps)                                 # [n_ring_steps + 1, 3] Patch-, [3] Point-
holonomy_steps = islice(zip(rings, parallel_transport(rings, starts)), None, None, ring_stride)
holonomy_states = show_states(
    holonomy_steps, render.holonomy_animation, sphere_topology, sphere_vertices, rings,
    name="connections_holonomy", duration_ms=frame_duration_ms,
)

Faint needles trail each tangent vector around its latitude loop. Along the green equator the enclosed hemisphere subtends a full turn of solid angle so the vector returns aligned with its initial meridian direction, along the blue sixty-degree ring the cap subtends half a turn so the vector returns reversed, and along the red thirty-degree ring near the north pole the smaller cap turns the vector by roughly fifty degrees.

## Covariant derivative and connection Laplacian

On a triangulated surface, an edge connection assigns a rotor to each endpoint of every edge carrying its tangent plane onto the head's tangent plane. For the surface's intrinsic Levi-Civita connection, the tail rotor is the dihedral turn `(heads * tails).square_root()` between endpoint patches and the head rotor is the identity. Taking the oriented boundary difference of endpoint vectors carried to each edge head defines the covariant derivative `(spdiag(transports >> Point) * topology.boundary[1, 0]).sum(axis=0)`, and pairing those edge differences against `Point.dual()` with cotangent weights before pulling back via `.adjugate()` builds the connection Laplacian `covariant.adjugate()((weights * Point.dual())(covariant))`. Because every edge rotor maps the tail patch onto the head patch, neither operator mixes tangent and normal components.

In [ ]:
# Torus major and minor radii and grid subdivisions around the hole and tube.
major_radius = 2.0
minor_radius = 0.75
n_heat_major = 28
n_heat_minor = 14

In [ ]:
def vertex_patches(topology: Topology, vertices: Point) -> Patch:
    """Joining two boundary edges on each triangle and pulling their area-weighted patches back to
    incident vertices gives the normalized vertex surface patches (`Patch-[V]`)."""
    edges = topology.boundary[2, 1] * (topology.d[0] * vertices)                          # [3] Point-[F]
    return (topology.incidence[2, 0].sum(axis=0).adjugate() * (edges[0] & edges[1])).normalized()


def vertex_areas(topology: Topology, vertices: Point) -> Scalar:
    """Distributing one third of each triangle's area (`(edges[0] & edges[1]).norm() / 2`) to its
    three corners gives the lumped vertex areas (`Scalar+[V]`)."""
    edges = topology.boundary[2, 1] * (topology.d[0] * vertices)                          # [3] Point-[F]
    corner_mean = topology.incidence[2, 0].mean(axis=0)                                   # Scalar+[F <- V]
    return corner_mean.adjugate() * ((edges[0] & edges[1]).norm() / 2)

In [ ]:
def cotangent_weights(topology: Topology, vertices: Point, face_weights: Scalar) -> Scalar:
    """Half-cotangents of opposite corner angles weighted by `face_weights` (`Scalar+[F]`) sum into
    the symmetric edge weights (`Scalar+[E]`)."""
    edges = topology.boundary[2, 1] * (topology.d[0] * vertices)                          # [3] Point-[F]
    tails, heads = topology.corners[2, 1]
    tail_edges, head_edges = edges[tails], edges[heads]                                   # [3] Point-[F], [3] Point-[F]
    cotangents = (tail_edges | head_edges) / (tail_edges & head_edges).norm()             # [3] Scalar+[F]
    return (topology.incidence[2, 1].adjugate() * (face_weights * cotangents / 2)).sum(axis=0)


def connection_operators(topology: Topology, transports: Rotor,
                         weights: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """Edge endpoint rotors `transports` (`[2] Rotor+[E]`) and edge `weights` (`Scalar+[E]`) assemble
    the sparse covariant derivative (`(Point- <- Point-)[E <- V]`) and connection Laplacian
    (`(Plane+ <- Point-)[V <- V]`)."""
    # Carry both endpoints' vectors into the head tangent plane and take their oriented difference.
    covariant = (spdiag(transports >> Point) * topology.boundary[1, 0]).sum(axis=0)       # (Point- <- Point-)[E <- V]
    # Pair each edge difference with its dual plane, weight by `weights`, and pull back via `.adjugate()`.
    laplacian = covariant.adjugate()((weights * Point.dual())(covariant))                 # (Plane+ <- Point-)[V <- V]
    return covariant, laplacian


torus_topology, torus_coordinates = shapes.torus(major_radius, minor_radius, n_heat_major, n_heat_minor)
V, E, F = torus_topology.n_elements
torus_vertices = mv.vector(torus_coordinates).dual().field()                              # Point-[V]
torus_patches = vertex_patches(torus_topology, torus_vertices)                            # Patch-[V]
torus_areas = vertex_areas(torus_topology, torus_vertices)                                # Scalar+[V]
torus_weights = cotangent_weights(torus_topology, torus_vertices, mv.scalar(np.ones((F, 1))).field())  # Scalar+[E]
edge_ends = torus_topology.incidence[1, 0] * torus_patches                                # [2] Patch-[E]
# Carry the tail tangent plane onto the head by the dihedral rotor while leaving the head fixed.
edge_turns = (edge_ends[1] * edge_ends).square_root()                                     # [2] Rotor+[E]
covariant, laplacian = connection_operators(torus_topology, edge_turns, torus_weights)

print(covariant.gatype, covariant.shape)
print(laplacian.gatype, laplacian.shape)

## Parallel transport by vector heat diffusion

Diffusing a localized tangent impulse on the outer equator by backward-Euler steps `(mass + step_time * laplacian).solve(mass(field))` carries its direction across the torus along all surface paths simultaneously. Normalizing the diffused vector at every vertex yields a smooth unit tangent field transported globally from the impulse.

In [ ]:
# Width of the initial tangent bump and duration and count of backward-Euler heat steps.
source_width = 0.55
step_time = 0.1
n_heat_steps = 36

In [ ]:
def heat_diffusion(laplacian: SparseExtensor, areas: Scalar, source: Point,
                   step_time: float, n_steps: int) -> Iterator[Point]:
    """Yields the diffused tangent vector field (`Point-[V]`) after each of `n_steps` backward-Euler
    vector heat steps of duration `step_time` from `source`."""
    mass = areas * Point.dual()                                                           # (Plane+ <- Point-)[V]
    step_operator = mass + step_time * laplacian                                          # (Plane+ <- Point-)[V <- V]
    field = source                                                                        # Point-[V]
    for _ in range(n_steps):
        field = step_operator.solve(mass(field))                                          # Point-[V]
        yield field


batch_patches = torus_patches.batch()                                                     # [V] Patch-
centre = (mv.x * (major_radius + minor_radius)).dual()                                    # [] Point-
bump = ((torus_vertices.batch() - centre).norm_squared() * (-0.5 / source_width**2)).exp()  # [V] Scalar+
direction = (mv.y + mv.z).dual().normalized()                                             # [] Point-
# Subtract the normal projection so the initial impulse lies strictly in each vertex's tangent plane.
tangent_direction = direction - batch_patches.dual() * (batch_patches & direction)        # [V] Point-
torus_source = (bump * tangent_direction).field()                                         # Point-[V]
diffusion_steps = show_states(
    heat_diffusion(laplacian, torus_areas, torus_source, step_time, n_heat_steps),
    render.diffusion_animation, torus_topology, torus_vertices,
    name="connections_diffusion", duration_ms=frame_duration_ms,
)
diffused_fields = stack(diffusion_steps)                                                  # [n_heat_steps] Point-[V]
transported_field = diffused_fields[-1] / diffused_fields[-1].norm()                      # Point-[V]

Symmetric surface streaks of uniform length trace the unit direction of the diffusing vector at every step regardless of local heat amplitude, spreading from the outer equator around both sides of the tube until the orientation settles across the torus.

## Harmonic tangent vector fields

Because ambient fields pointing along the surface normals can also have small covariant variation, adding a stiff quadratic penalty on normal components `unit_patches.dual() * (unit_patches & Point)` lifts the normal bundle out of the low spectrum while leaving tangent fields untouched. Solving the generalized eigenproblem `(laplacian + normal_penalty).eigh(spdiag(areas * Point.dual()), n_modes)` then extracts the smoothest global tangent fields of any connection Laplacian. Because a quarter turn in every tangent plane `field.commutator(patches.dual())` commutes with every edge rotor, eigenvalues always occur in equal-energy orthogonal pairs.

In [ ]:
# Stiffness lifting normal components out of the low spectrum and number of eigenmodes computed.
normal_stiffness = 100.0
n_modes = 4

In [ ]:
def low_modes(laplacian: SparseExtensor, patches: Patch, areas: Scalar,
              normal_stiffness: float, n_modes: int) -> tuple[Scalar, Point]:
    """Penalizing normal components by `normal_stiffness` lifts the normal bundle out of the low
    spectrum and solves for the `n_modes` smoothest tangent eigenfields (`[n_modes] Point-[V]`) and
    their Dirichlet energies (`[n_modes] Scalar+`) against vertex `areas`."""
    unit_patches = patches.batch()                                                        # [V] Patch-
    # Project each direction onto the surface normal direction and pair with its dual plane.
    normal_part = (unit_patches.dual() * (unit_patches & Point)).field()                  # (Point- <- Point-)[V]
    normal_penalty = normal_stiffness * Point.dual()(normal_part)                         # (Plane+ <- Point-)[V]
    return (laplacian + normal_penalty).eigh(spdiag(areas * Point.dual()), n_modes)


mode_energies, modes = low_modes(laplacian, torus_patches, torus_areas, normal_stiffness, n_modes)

save_figure(render.torus_fields_figure(torus_topology, torus_vertices,
                                       [transported_field, modes[0], modes[2]]), "connections_modes");

From left to right, the panels show the normalized vector-heat field diffused from the outer-equator impulse, the lowest-energy harmonic tangent mode winding smoothly around the torus, and a mode from the next degenerate pair.

## Face curvature as loop holonomy

On a closed surface other than a torus, every continuous tangent vector field vanishes or spins at isolated singularities whose winding indices sum to the Euler characteristic, which equals two on a sphere and zero on a torus. Carrying a tangent vector around the three corners of a single triangle rotates it in its starting tangent plane by the triangle's intrinsic curvature holonomy. Because a full rotation flips the sign of a spinor rotor, squaring the cumulative loop rotor reads the signed holonomy angle `(loop_rotor * loop_rotor).log() | patches[0].dual()` unambiguously between minus and plus half a turn, and summing those triangle holonomies over a closed sphere recovers two full turns by the discrete Gauss–Bonnet theorem.

In [ ]:
def loop_holonomy(patches: Patch) -> Scalar:
    """Chaining dihedral transport rotors around closed loops of unit `patches` (`[n_steps, ...] Patch-`)
    reads each loop's signed holonomy angle (`[...] Scalar+`) in `(-pi, pi]` from the squared loop rotor."""
    next_patches = concatenate([patches[1:], patches[:1]])                                # [n_steps, ...] Patch-
    step_rotors = (next_patches * patches).square_root()                                  # [n_steps, ...] Rotor+
    loop_rotor = step_rotors.cumprod(axis=0)[-1]                                          # [...] Rotor+
    return (loop_rotor * loop_rotor).log() | patches[0].dual()                            # [...] Scalar+


sphere_patches = vertex_patches(sphere_topology, sphere_vertices)                         # Patch-[V_sphere]
sphere_areas = vertex_areas(sphere_topology, sphere_vertices)                             # Scalar+[V_sphere]
sphere_weights = cotangent_weights(sphere_topology, sphere_vertices, mv.scalar(np.ones((F_sphere, 1))).field())  # Scalar+[E_sphere]
corner_patches = (sphere_topology.incidence[2, 0] * sphere_patches).batch()               # [3, F_sphere] Patch-
curvature = loop_holonomy(corner_patches)                                                 # [F_sphere] Scalar+

print("Total triangle holonomy in whole turns:", (curvature.sum(axis=0) / (2 * np.pi)).to_array())

## Trivial connections from edge turns

A connection becomes trivial outside chosen singular triangles when carrying a tangent vector around every regular loop returns it unchanged. Rotating each edge's transported vector in the head tangent plane by an extra angle `(heads.dual() * (-angles / 2)).exp()` shifts every loop's holonomy linearly by the signed edge circulation around that loop. Requiring the edge circulation around each triangle to match its target singularity turn minus its intrinsic curvature, while cancelling the surface's holonomy around any topological handle loops, forms an underdetermined linear system whose minimum-norm solution `circulation.lstsq(target)` twists the connection as gently as possible.

In [ ]:
# Icosahedral vertex indices whose incident triangles carry singularities, paired with their topological indices.
sphere_corners = [[9, 10], [9], [9, 4, 8, 3]]
sphere_indices = [[1, 1], [2], [1, 1, 1, -1]]

In [ ]:
def adjusted_transport(topology: Topology, patches: Patch, angles: Scalar) -> Rotor:
    """Composing each edge's dihedral rotor from tail to head with an extra tangent-plane rotation by
    `angles` (`Scalar+[E]`) forms the adjusted edge endpoint rotors (`[2] Rotor+[E]`)."""
    tails, heads = topology.incidence[1, 0] * patches                                     # Patch-[E], Patch-[E]
    extra_turns = (heads.dual() * (-angles / 2)).exp()                                    # Rotor+[E]
    edge_rotors = extra_turns * (heads * tails).square_root()                             # Rotor+[E]
    return stack([edge_rotors, mv.rotor().broadcast_to(edge_rotors.batch().shape).field()])  # [2] Rotor+[E]


def trivial_connection(topology: Topology, patches: Patch, indices: np.ndarray,
                       generators: list[np.ndarray]) -> tuple[Scalar, SparseExtensor, Scalar]:
    """Least-squares edge turns (`Scalar+[E]`) cancel face curvature up to `2 * pi * indices` on
    triangles and cancel handle holonomy around `generators`, returning the turns alongside the
    sparse circulation operator and target holonomy field."""
    corner_patches = (topology.incidence[2, 0] * patches).batch()                         # [3, F] Patch-
    loop_patches = [(topology.reading(loop[:, None], 0) * patches).batch() for loop in generators]
    curvature = loop_holonomy(corner_patches)                                             # [F] Scalar+
    handle_targets = [-loop_holonomy(handle_patches) for handle_patches in loop_patches]
    circulation = topology.circulation(generators) * Scalar                               # (Scalar+ <- Scalar+)[F_loops <- E]
    target = concatenate([2 * np.pi * indices - curvature, *handle_targets]).field()      # Scalar+[F_loops]
    return circulation.lstsq(target), circulation, target


sphere_singular_faces = [sphere_topology.vertex_simplices[corners] for corners in sphere_corners]
sphere_designs = [
    trivial_connection(sphere_topology, sphere_patches, face_indices(sphere_topology, singular_faces, indices), [])
    for singular_faces, indices in zip(sphere_singular_faces, sphere_indices)
]
sphere_residuals = [circulation(angles) - target for angles, circulation, target in sphere_designs]

## Parallel tangent vector fields

Once edge turns make the connection trivial on all regular loops, transporting a single tangent vector across the surface agrees along every path. Consequently, the adjusted connection Laplacian possesses a two-dimensional zero-energy null space spanned by a globally parallel unit tangent field and its tangent-plane quarter turn.

In [ ]:
def parallel_field(topology: Topology, patches: Patch, areas: Scalar, weights: Scalar,
                   angles: Scalar, normal_stiffness: float) -> tuple[Scalar, Point]:
    """Extracting the two lowest eigenmodes of the adjusted connection with edge turns `angles`
    returns their energies (`[2] Scalar+`) and the unit parallel tangent field (`Point-[V]`)."""
    _, laplacian = connection_operators(topology, adjusted_transport(topology, patches, angles), weights)
    energies, fields = low_modes(laplacian, patches, areas, normal_stiffness, 2)
    field = fields.batch()[0]                                                             # [V] Point-
    return energies, (field / field.norm()).field()


sphere_energies, sphere_fields = zip(*[
    parallel_field(sphere_topology, sphere_patches, sphere_areas, sphere_weights, angles, normal_stiffness)
    for angles, _, _ in sphere_designs
])
sphere_scenes = list(zip(sphere_fields, sphere_singular_faces, sphere_indices))

save_figure(render.fields_figure(sphere_topology, sphere_vertices, sphere_scenes), "connections_fields_spheres");

Directed surface streaks fade downstream to distinguish sources from sinks across the three designed fields. The left sphere places two index-one singularities at antipodal poles with one visible in front, the middle sphere concentrates all turning into a single index-two dipole singularity, and the right sphere surrounds an index-minus-one saddle with three index-one singularities.

## Singularity indices and the Euler characteristic

Summing the edge circulation over every triangle traverses each interior edge once in each direction and cancels identically, so a trivial connection exists only when the requested singularity turns sum to the total Gaussian curvature of the surface. This solvability condition expresses the Poincaré–Hopf theorem directly on the complex. Requesting an index sum one short of the sphere's Euler characteristic leaves one whole turn of holonomy unmet, which the least-squares solve distributes uniformly across every triangle.

In discrete exterior calculus notation, the edge turns form a primal 1-cochain $\theta \in C^1$, the face constraints read $d\theta = 2\pi k - \Omega$, and exactness on a closed surface requires $\sum_f k_f = \frac{1}{2\pi}\sum_f \Omega_f = \chi$.

In [ ]:
unmet_angles, unmet_circulation, unmet_target = trivial_connection(
    sphere_topology, sphere_patches, face_indices(sphere_topology, np.array([0]), [1]), [],
)
unmet_holonomy = unmet_circulation(unmet_angles) - unmet_target                           # Scalar+[F_sphere]

print("Total unmet holonomy in whole turns:", (unmet_holonomy.sites.sum() / (2 * np.pi)).to_array())

## Handle holonomy on the torus

Because the Euler characteristic of a torus vanishes, a tangent vector field can cover the entire surface with zero singularities, or carry opposite-index pairs whose indices sum to zero. Vanishing triangle holonomy alone leaves global transport around the tube and around the central hole unconstrained, so two additional rows in `circulation` cancel the surface's intrinsic holonomy around a meridian generator and a longitudinal generator.

In [ ]:
# Torus grid dimensions and singular triangle coordinates for singularity-free and dipole designs.
n_field_major, n_field_minor = 40, 20
field_torus_places = [np.zeros((0, 2), dtype=int), np.array([[32, 2], [0, 2]])]
torus_indices = [[], [1, -1]]

In [ ]:
ring_topology, ring_coordinates = shapes.torus(major_radius, minor_radius, n_field_major, n_field_minor)
V_ring, E_ring, F_ring = ring_topology.n_elements
ring_vertices = mv.vector(ring_coordinates).dual().field()                                # Point-[V_ring]
ring_patches = vertex_patches(ring_topology, ring_vertices)                               # Patch-[V_ring]
ring_areas = vertex_areas(ring_topology, ring_vertices)                                   # Scalar+[V_ring]
ring_weights = cotangent_weights(ring_topology, ring_vertices, mv.scalar(np.ones((F_ring, 1))).field())  # Scalar+[E_ring]
# One vertex loop around the minor tube and one around the outer equator.
generators = [np.arange(n_field_minor), np.arange(n_field_major) * n_field_minor]
torus_singular_faces = [places[:, 0] * n_field_minor + places[:, 1] for places in field_torus_places]
torus_designs = [
    trivial_connection(ring_topology, ring_patches, face_indices(ring_topology, singular_faces, indices), generators)
    for singular_faces, indices in zip(torus_singular_faces, torus_indices)
]
torus_residuals = [circulation(angles) - target for angles, circulation, target in torus_designs]
torus_energies, torus_fields = zip(*[
    parallel_field(ring_topology, ring_patches, ring_areas, ring_weights, angles, normal_stiffness)
    for angles, _, _ in torus_designs
])
torus_scenes = list(zip(torus_fields, torus_singular_faces, torus_indices))

save_figure(render.fields_figure(ring_topology, ring_vertices, torus_scenes), "connections_fields_tori");

The left torus carries a globally parallel tangent field free of singularities, while the right torus places an index-one source and an index-minus-one saddle on the front face.

## Turning a parallel field

Because the trivial connection's null space is closed under uniform tangent-plane rotations, rotating a parallel field by the same angle `(patches.dual() * (-angle / 2)).exp()` at every vertex sweeps through the entire family of parallel fields sharing the same connection.

In [ ]:
# Frame count for one full rotation through the parallel null space.
n_turning_frames = 84

In [ ]:
def turned_fields(patches: Patch, fields: Point, n_frames: int) -> Iterator[Point]:
    """Yields `fields` (`[...] Point-[V]`) rotated about every vertex normal by `n_frames` evenly
    spaced angles over one full turn."""
    planes = patches.dual()                                                               # Bivector+[V]
    for angle in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        turn = (planes * (-angle / 2)).exp()                                              # Rotor+[V]
        yield turn >> fields                                                              # [...] Point-[V]


first_field, _, _ = sphere_scenes[0]
turned_states = show_states(
    turned_fields(sphere_patches, first_field, n_turning_frames),
    render.turning_animation, sphere_topology, sphere_vertices,
    name="connections_turning", duration_ms=frame_duration_ms,
)

As the field rotates through a full turn, the pattern around the visible index-one singularity transitions smoothly from outward radial flow through clockwise spiralling to inward flow and counter-clockwise circulation.

## Stripe patterns

A stripe pattern represents a periodic scalar phase whose bands run perpendicular to a guiding parallel field `field` at spatial `frequency`. Measuring the phase at each vertex as the tangent-plane angle from `field` to a unit tangent vector `stripe`, advancing that phase along every edge adds a directional turn `(-((edge_means * field) | (topology.d[0] * vertices))) * frequency` onto the trivial connection's edge angles. Finding the lowest eigenmode of `connection_operators` with those augmented turns, while zeroing cotangent weights on singular triangles where `field` winds rapidly, yields a globally smooth stripe field.

In [ ]:
# Subdivision level of the finer sphere and spatial frequency of the stripe pattern.
n_stripe_levels = 4
sphere_frequency = 20.0

In [ ]:
def stripe_family(topology: Topology, vertices: Point, patches: Patch, areas: Scalar,
                  angles: Scalar, field: Point, regular_faces: Scalar, frequency: float,
                  normal_stiffness: float) -> tuple[Point, Point, Scalar, Scalar]:
    """Adding directional phase advances of `frequency` along `field` (`Point-[V]`) on regular faces
    (`regular_faces`, `Scalar+[F]`) solves for the smoothest unit stripe field (`Point-[V]`), its edge
    advances (`Scalar+[E]`), and the lowest two stripe energies (`[2] Scalar+`)."""
    edge_means = topology.incidence[1, 0].mean(axis=0)                                    # Scalar+[E <- V]
    advance = (-((edge_means * field) | (topology.d[0] * vertices))) * frequency          # Scalar+[E]
    _, laplacian = connection_operators(
        topology, adjusted_transport(topology, patches, angles + advance),
        cotangent_weights(topology, vertices, regular_faces),
    )
    energies, stripe_fields = low_modes(laplacian, patches, areas, normal_stiffness, 2)
    stripe = stripe_fields.batch()[0]                                                     # [V] Point-
    return field, (stripe / stripe.norm()).field(), advance, energies


fine_topology, fine_coordinates = shapes.sphere(n_stripe_levels)
V_fine, E_fine, F_fine = fine_topology.n_elements
fine_vertices = mv.vector(fine_coordinates).dual().field()                                # Point-[V_fine]
fine_patches = vertex_patches(fine_topology, fine_vertices)                               # Patch-[V_fine]
fine_areas = vertex_areas(fine_topology, fine_vertices)                                   # Scalar+[V_fine]
fine_weights = cotangent_weights(fine_topology, fine_vertices, mv.scalar(np.ones((F_fine, 1))).field())  # Scalar+[E_fine]
fine_singular_faces = [fine_topology.vertex_simplices[corners] for corners in sphere_corners]
fine_indices = [face_indices(fine_topology, singular_faces, indices)
                for singular_faces, indices in zip(fine_singular_faces, sphere_indices)]
fine_angles = [trivial_connection(fine_topology, fine_patches, index, [])[0] for index in fine_indices]
fine_fields = [parallel_field(fine_topology, fine_patches, fine_areas, fine_weights, angles, normal_stiffness)[1]
               for angles in fine_angles]
stripe_runs = [
    stripe_family(fine_topology, fine_vertices, fine_patches, fine_areas, angles, field,
                  mv.scalar(np.where(index == 0, 1.0, 0.0)[:, None]).field(), sphere_frequency, normal_stiffness)
    for index, angles, field in zip(fine_indices, fine_angles, fine_fields)
]
stripe_patterns = [(line[None], stripe[None], advance[None]) for line, stripe, advance, _ in stripe_runs]
stripe_energies = [energies for *_, energies in stripe_runs]

save_figure(render.bands_figure(fine_topology, fine_vertices, stripe_patterns), "connections_stripes_spheres");

Alternating light and dark bands trace the unwrapped angle between the stripe field and the guiding parallel field across the three sphere singularity configurations. Wherever surface curvature causes neighbouring bands to converge or diverge beyond uniform spacing, the lowest eigenmode lets its amplitude vanish inside a single triangle so a stripe terminates or forks cleanly.

## Cross-field stripes

A cross field assigns two orthogonal unoriented tangent lines to every vertex and permits quarter-turn and half-turn singularities whose fractional indices still sum to the Euler characteristic. Propagating a unit tangent seed across a breadth-first spanning tree under quarter-turn or half-turn trivial-connection angles produces a representative cross field `cross` whose transport across every non-tree edge lands aligned with or opposite to one of the two orthogonal lines `[cross, cross.commutator(patches.dual())]`. Because reversing a guiding line negates phases measured from it, coupling both stripe families into a `2 * V` vertex system that swaps and mirrors lines across edges produces globally coherent two-family cross-stripe grids.

In [ ]:
# Cube-corner (+1/4) and inscribed-tetrahedron (+1/2) cross-field singularity directions and panel scale.
cube_directions = [[1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
                   [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]]
cross_corners = [cube_directions, [[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]]
cross_indices = [0.25, 0.5]
cross_scale = 0.5

In [ ]:
def spanning_tree_field(topology: Topology, vertices: Point, patches: Patch, angles: Scalar) -> Point:
    """Propagating a unit tangent seed from the first vertex across `topology.spanning_tree` under
    `angles` (`Scalar+[E]`) in one sparse linear solve produces a unit cross-field representative
    (`Point-[V]`) parallel up to quarter turns across non-tree edges."""
    children, tree = topology.spanning_tree
    turns = adjusted_transport(topology, patches, angles)                                 # [2] Rotor+[E]
    covariant = (spdiag(turns >> Point) * topology.boundary[1, 0]).sum(axis=0)            # (Point- <- Point-)[E <- V]
    tree_edges = topology.reading(tree, 1)                                                # Scalar+[V_tree <- E]
    tree_covariant = topology.reading(children, 0).adjugate() * (tree_edges * covariant)  # (Point- <- Point-)[V <- V]
    root_vertex = topology.reading(np.array([0]), 0)                                      # Scalar+[1 <- V]
    first_edge = topology.reading(np.array([0]), 1)                                       # Scalar+[1 <- E]
    # Turn the first edge vector in the root vertex's tangent plane to obtain an initial unit tangent seed.
    seed = (-(first_edge * (topology.d[0] * vertices)).commutator((root_vertex * patches).dual())).normalized()  # Point-[1]
    return (tree_covariant + root_vertex.adjugate() * root_vertex * Point).solve(root_vertex.adjugate() * seed)


def cross_stripes(topology: Topology, vertices: Point, patches: Patch, areas: Scalar,
                  angles: Scalar, cross: Point, regular_faces: Scalar, frequency: float,
                  normal_stiffness: float) -> tuple[Point, Point, Scalar]:
    """Coupling the two orthogonal cross-field lines (`[2] Point-[V]`) across edges where quarter or
    half turns swap or reverse them solves for both coupled unit stripe families (`[2] Point-[V]`)
    and their edge phase advances (`[2] Scalar+[E]`)."""
    V, E, F = topology.n_elements
    edge_rotors = adjusted_transport(topology, patches, angles)[0]                        # Rotor+[E]
    # Turn each cross line by a quarter turn in its tangent plane via commutator with the dual patch.
    lines = stack([cross, cross.commutator(patches.dual())])                              # [2] Point-[V]
    tails, heads = topology.incidence[1, 0]                                               # Scalar+[E <- V], Scalar+[E <- V]
    tail_lines, head_lines = tails * lines, heads * lines                                 # [2] Point-[E], [2] Point-[E]
    transported_lines = edge_rotors >> tail_lines                                         # [2] Point-[E]
    # Signed alignment (+1, -1, or 0) matching each transported tail line onto each head line.
    alignment = -(head_lines[:, None] | transported_lines[None, :])                       # [2, 2] Scalar+[E]
    matched_tails = (alignment * tail_lines[None, :]).sum(axis=1)                         # [2] Point-[E]
    advance = (-((head_lines + matched_tails) / 2 | (topology.d[0] * vertices))) * frequency  # [2] Scalar+[E]
    target_turn = ((heads * patches).dual() * (-advance / 2)).exp()                       # [2] Rotor+[E]
    # Mirror the carried stripe vector along the head line whenever transport lands against that line.
    transport_map = edge_rotors >> Point                                                  # (Point- <- Point-)[E]
    head_axis = head_lines[:, None]                                                       # [2, 1] Point-[E]
    coupled_transport = (alignment * alignment * transport_map
                         + (alignment - alignment * alignment) * (head_axis * -(head_axis | transport_map)))  # [2, 2] (Point- <- Point-)[E]
    vertex_lines = Sparse.selection(np.arange(2 * V).reshape(V, 2).T, 2 * V)              # [2] Scalar+[V <- 2 * V]
    edge_lines = Sparse.selection(np.arange(2 * E).reshape(E, 2).T, 2 * E)                # [2] Scalar+[E <- 2 * E]
    line_differences = (spdiag(target_turn.reverse() >> Point) * heads * vertex_lines
                        - (spdiag(coupled_transport) * tails * vertex_lines[None]).sum(axis=1))  # [2] (Point- <- Point-)[E <- 2 * V]
    covariant = (edge_lines.adjugate() * line_differences).sum(axis=0)                    # (Point- <- Point-)[2 * E <- 2 * V]
    weights = (edge_lines.adjugate() * cotangent_weights(topology, vertices, regular_faces)).sum(axis=0)  # Scalar+[2 * E]
    laplacian = covariant.adjugate()((weights * Point.dual())(covariant))                 # (Plane+ <- Point-)[2 * V <- 2 * V]
    _, stripe_fields = low_modes(
        laplacian, (vertex_lines.adjugate() * patches).sum(axis=0),
        (vertex_lines.adjugate() * areas).sum(axis=0), normal_stiffness, 1,
    )
    stripe = vertex_lines * stripe_fields[0]                                              # [2] Point-[V]
    return lines, stripe / stripe.norm(), advance


cross_singular_indices = [
    face_indices(fine_topology, nearest_faces(fine_topology, fine_vertices, mv.vector(corners)), index)
    for corners, index in zip(cross_corners, cross_indices)
]
cross_angles = [trivial_connection(fine_topology, fine_patches, index, [])[0] for index in cross_singular_indices]
cross_patterns = [
    cross_stripes(
        fine_topology, fine_vertices, fine_patches, fine_areas, angles,
        spanning_tree_field(fine_topology, fine_vertices, fine_patches, angles),
        mv.scalar(np.where(index == 0, 1.0, 0.0)[:, None]).field(),
        sphere_frequency, normal_stiffness,
    )
    for index, angles in zip(cross_singular_indices, cross_angles)
]

save_figure(render.lines_figure(fine_topology, fine_vertices, cross_patterns, cross_scale), "connections_stripes_crosses");

Zero-level stripe centrelines trace two orthogonal grids across the sphere. The left sphere places eight quarter-turn singularities at the corners of an inscribed cube where three grid squares meet around each corner, while the right sphere places four half-turn singularities at the corners of an inscribed tetrahedron where the grid folds back around U-turn defects.

In [ ]:
# checks
# Latitude-ring transport steps stay orthogonal to the sphere patches, keep unit norm, and turn by the cap solid angle.
sampled_patches, carried_tangents = map(stack, zip(*holonomy_states))                     # [..., 3] Patch-, [..., 3] Point-
assert ((carried_tangents & sampled_patches).antinorm() < 1e-10).all()
assert ((carried_tangents.norm() - 1).abs() < 1e-10).all()
expected_cosines = mv.scalar(np.cos(2 * np.pi * (1 - np.cos(colatitudes)))[:, None])      # [3] Scalar+
assert ((-(carried_tangents[-1] | starts) - expected_cosines).abs() < 2e-3).all()

# Vector heat diffusion, harmonic eigenmodes, and turned parallel frames remain strictly tangent.
assert ((diffused_fields & torus_patches).batch().antinorm() < 1e-10).all()
assert ((modes & torus_patches).batch().antinorm() < 1e-10).all()
assert ((stack(turned_states) & sphere_patches).batch().antinorm() < 1e-10).all()

# Tangent-plane quarter turns commute with the connection, giving equal-energy mode pairs.
assert ((mode_energies[0::2] - mode_energies[1::2]).abs() < 1e-9).all()
edge_weights = spdiag(torus_weights)                                                      # Scalar+[E <- E]
quarter_turn_modes = modes.batch().commutator(torus_patches.batch().dual()).field()       # [n_modes] Point-[V]
mode_dirichlet = (edge_weights * covariant(modes).norm_squared()).sites.sum()             # [n_modes] Scalar+
turned_dirichlet = (edge_weights * covariant(quarter_turn_modes).norm_squared()).sites.sum()  # [n_modes] Scalar+
assert ((turned_dirichlet - mode_dirichlet).abs() < 1e-9).all()

# Triangle holonomies on the sphere sum to 4 * pi, while an index sum short by 1 spreads 2 * pi / F_sphere uniformly.
assert ((curvature.sum(axis=0) - 4 * np.pi).abs() < 1e-10).all()
assert ((unmet_holonomy.batch() - 2 * np.pi / F_sphere).abs() < 1e-8).all()

# Valid trivial connections satisfy circulation exactly and support zero-energy parallel tangent fields.
all_residuals = concatenate([residual.batch() for residual in sphere_residuals + torus_residuals])
all_energies = concatenate([*sphere_energies, *torus_energies])
assert (all_residuals.abs() < 1e-10).all()
assert (all_energies.abs() < 1e-9).all()

# Stripe modes come in equal-energy pairs and follow their prescribed edge phase advance away from forks.
for (pat_lines, pat_stripes, pat_advance), pair_energies in zip(stripe_patterns, stripe_energies):
    assert ((pair_energies[1] - pair_energies[0]).abs() < 1e-5 * (pair_energies[0].abs() + 1)).all()
    phase_rotors = (pat_stripes * pat_lines.reverse()).normalized()                       # [1] Rotor+[V_fine]
    phases = phase_rotors.log().scalar_product(fine_patches.dual())                       # [1] Scalar+[V_fine]
    steps = (fine_topology.d[0] * phases - pat_advance).batch()                           # [1, E_fine] Scalar+
    mismatch = ((mv.xy * steps).exp().log() | mv.xy).abs()                                # [1, E_fine] Scalar+
    assert ((mismatch < 0.25).sum(axis=-1) > E_fine // 2).all()

# Cross-field lines map onto each other up to sign across every edge and swap nontrivially around singularities.
for (cross_lines, _, _), angles in zip(cross_patterns, cross_angles):
    edge_rotors = adjusted_transport(fine_topology, fine_patches, angles)[0]              # Rotor+[E_fine]
    tails, heads = fine_topology.incidence[1, 0][:, None] * cross_lines                   # [2] Point-[E_fine], [2] Point-[E_fine]
    transported_lines = edge_rotors >> tails                                              # [2] Point-[E_fine]
    alignment = (-(heads[:, None] | transported_lines[None, :])).batch()                  # [2, 2, E_fine] Scalar+
    assert ((alignment.abs() * (1 - alignment.abs())).abs() < 1e-7).all()
    assert (alignment[0, 0] < 0.5).any()